In [16]:
!pip install matplotlib imutils keras scikit-learn


In [17]:
import os
import random
import shutil
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from imutils import paths
from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import LearningRateScheduler
from keras.optimizers import Adagrad
from keras.utils import np_utils
from keras.models import Sequential
from keras.layers import BatchNormalization
from keras.layers.convolutional import SeparableConv2D, MaxPooling2D
from keras.layers.core import Activation, Flatten, Dropout, Dense
from keras import backend as K
from sklearn.metrics import classification_report, confusion_matrix

In [18]:
# Configuration
INPUT_DATASET = r"D:\1A Shiash\Projects\Breast Cancer\DataSet"
BASE_PATH = r"D:\1A Shiash\Projects\Breast Cancer"
TRAIN_PATH = os.path.join(BASE_PATH, "training")
VAL_PATH = os.path.join(BASE_PATH, "validation")
TEST_PATH = os.path.join(BASE_PATH, "testing")
TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.1

In [19]:
# Training parameters
NUM_EPOCHS = 5
INIT_LR = 1e-2
BS = 32


In [20]:
class CancerNet:
    @staticmethod
    def build(width, height, depth, classes):
        model = Sequential()
        shape = (height, width, depth)
        channelDim = -1

        if K.image_data_format() == "channels_first":
            shape = (depth, height, width)
            channelDim = 1

        model.add(SeparableConv2D(32, (3, 3), padding="same", input_shape=shape))
        model.add(Activation("relu"))
        model.add(BatchNormalization(axis=channelDim))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(Dropout(0.25))

        model.add(SeparableConv2D(64, (3, 3), padding="same"))
        model.add(Activation("relu"))
        model.add(BatchNormalization(axis=channelDim))
        model.add(SeparableConv2D(64, (3, 3), padding="same"))
        model.add(Activation("relu"))
        model.add(BatchNormalization(axis=channelDim))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(Dropout(0.25))

        model.add(SeparableConv2D(128, (3, 3), padding="same"))
        model.add(Activation("relu"))
        model.add(BatchNormalization(axis=channelDim))
        model.add(SeparableConv2D(128, (3, 3), padding="same"))
        model.add(Activation("relu"))
        model.add(BatchNormalization(axis=channelDim))
        model.add(SeparableConv2D(128, (3, 3), padding="same"))
        model.add(Activation("relu"))
        model.add(BatchNormalization(axis=channelDim))
        model.add(MaxPooling2D(pool_size=(2, 2)))
        model.add(Dropout(0.25))

        model.add(Flatten())
        model.add(Dense(256))
        model.add(Activation("relu"))
        model.add(BatchNormalization())
        model.add(Dropout(0.5))

        model.add(Dense(classes))
        model.add(Activation("softmax"))

        return model

In [21]:
def build_dataset():
    originalPaths = list(paths.list_images(INPUT_DATASET))
    random.seed(7)
    random.shuffle(originalPaths)

    index = int(len(originalPaths) * TRAIN_SPLIT)
    trainPaths = originalPaths[:index]
    testPaths = originalPaths[index:]

    index = int(len(trainPaths) * VAL_SPLIT)
    valPaths = trainPaths[:index]
    trainPaths = trainPaths[index:]

    datasets = [
        ("training", trainPaths, TRAIN_PATH),
        ("validation", valPaths, VAL_PATH),
        ("testing", testPaths, TEST_PATH)
    ]

    for (setType, originalPaths, basePath) in datasets:
        print(f'Building {setType} set')

        if not os.path.exists(basePath):
            print(f'Building directory {basePath}')
            os.makedirs(basePath)

        for path in originalPaths:
            file = path.split(os.path.sep)[-1]
            label = file[-5:-4]

            labelPath = os.path.join(basePath, label)
            if not os.path.exists(labelPath):
                print(f'Building directory {labelPath}')
                os.makedirs(labelPath)

            newPath = os.path.join(labelPath, file)
            shutil.copy2(path, newPath)

In [22]:
def train_and_evaluate_model():
    trainPaths = list(paths.list_images(TRAIN_PATH))
    lenTrain = len(trainPaths)
    lenVal = len(list(paths.list_images(VAL_PATH)))
    lenTest = len(list(paths.list_images(TEST_PATH)))

    trainLabels = [int(p.split(os.path.sep)[-2]) for p in trainPaths]
    trainLabels = np_utils.to_categorical(trainLabels)
    classTotals = trainLabels.sum(axis=0)
    
    # Modify class weight calculation
    classWeight = {i: classTotals.max() / classTotals[i] for i in range(len(classTotals))}

    trainAug = ImageDataGenerator(
        rescale=1/255.0,
        rotation_range=20,
        zoom_range=0.05,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=0.05,
        horizontal_flip=True,
        vertical_flip=True,
        fill_mode="nearest")

    valAug = ImageDataGenerator(rescale=1 / 255.0)

    trainGen = trainAug.flow_from_directory(
        TRAIN_PATH,
        class_mode="categorical",
        target_size=(48, 48),
        color_mode="rgb",
        shuffle=True,
        batch_size=BS)
    valGen = valAug.flow_from_directory(
        VAL_PATH,
        class_mode="categorical",
        target_size=(48, 48),
        color_mode="rgb",
        shuffle=False,
        batch_size=BS)
    testGen = valAug.flow_from_directory(
        TEST_PATH,
        class_mode="categorical",
        target_size=(48, 48),
        color_mode="rgb",
        shuffle=False,
        batch_size=BS)

    model = CancerNet.build(width=48, height=48, depth=3, classes=2)
    opt = Adagrad(lr=INIT_LR, decay=INIT_LR / NUM_EPOCHS)
    model.compile(loss="binary_crossentropy", optimizer=opt, metrics=["accuracy"])

    # Update to use model.fit instead of model.fit_generator
    M = model.fit(
        trainGen,
        steps_per_epoch=lenTrain // BS,
        validation_data=valGen,
        validation_steps=lenVal // BS,
        class_weight=classWeight,
        epochs=NUM_EPOCHS)

    print("Now evaluating the model")
    testGen.reset()
    pred_indices = model.predict(testGen, steps=(lenTest // BS) + 1)

    pred_indices = np.argmax(pred_indices, axis=1)

    print(classification_report(testGen.classes, pred_indices, target_names=testGen.class_indices.keys()))

    cm = confusion_matrix(testGen.classes, pred_indices)
    total = sum(sum(cm))
    accuracy = (cm[0, 0] + cm[1, 1]) / total
    specificity = cm[1, 1] / (cm[1, 0] + cm[1, 1])
    sensitivity = cm[0, 0] / (cm[0, 0] + cm[0, 1])
    print(cm)
    print(f'Accuracy: {accuracy}')
    print(f'Specificity: {specificity}')
    print(f'Sensitivity: {sensitivity}')

    N = NUM_EPOCHS
    plt.style.use("ggplot")
    plt.figure()
    plt.plot(np.arange(0, N), M.history["loss"], label="train_loss")
    plt.plot(np.arange(0, N), M.history["val_loss"], label="val_loss")
    plt.plot(np.arange(0, N), M.history["accuracy"], label="train_acc")  # Changed from "acc" to "accuracy"
    plt.plot(np.arange(0, N), M.history["val_accuracy"], label="val_acc")  # Changed from "val_acc" to "val_accuracy"
    plt.title("Training Loss and Accuracy on the IDC Dataset")
    plt.xlabel("Epoch No.")
    plt.ylabel("Loss/Accuracy")
    plt.legend(loc="lower left")
    plt.savefig('plot.png')


In [23]:
if __name__ == "__main__":
    build_dataset()
    train_and_evaluate_model()


Building training set
Building directory D:\1A Shiash\Projects\Breast Cancer\training
Building directory D:\1A Shiash\Projects\Breast Cancer\training\0
Building directory D:\1A Shiash\Projects\Breast Cancer\training\1
Building validation set
Building directory D:\1A Shiash\Projects\Breast Cancer\validation
Building directory D:\1A Shiash\Projects\Breast Cancer\validation\0
Building directory D:\1A Shiash\Projects\Breast Cancer\validation\1
Building testing set
Building directory D:\1A Shiash\Projects\Breast Cancer\testing
Building directory D:\1A Shiash\Projects\Breast Cancer\testing\1
Building directory D:\1A Shiash\Projects\Breast Cancer\testing\0
Found 255815 images belonging to 2 classes.
Found 42660 images belonging to 2 classes.
Found 99906 images belonging to 2 classes.


c:\Users\srini\anaconda3\envs\Ten\lib\site-packages\keras\optimizers\optimizer_v2\adagrad.py:81: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super().__init__(name, **kwargs)


Epoch 1/5
7994/7994 [==============================] - 383s 47ms/step - loss: 0.6474 - accuracy: 0.8044 - val_loss: 1.1120 - val_accuracy: 0.5230
Epoch 2/5
7994/7994 [==============================] - 367s 46ms/step - loss: 0.6233 - accuracy: 0.8108 - val_loss: 1.1124 - val_accuracy: 0.5237
Epoch 3/5
7994/7994 [==============================] - 327s 41ms/step - loss: 0.6186 - accuracy: 0.8125 - val_loss: 1.0958 - val_accuracy: 0.5347
Epoch 4/5
7994/7994 [==============================] - 2000s 250ms/step - loss: 0.6156 - accuracy: 0.8129 - val_loss: 1.0724 - val_accuracy: 0.5474
Epoch 5/5
7994/7994 [==============================] - 324s 41ms/step - loss: 0.6138 - accuracy: 0.8145 - val_loss: 1.0632 - val_accuracy: 0.5489
Now evaluating the model
3123/3123 [==============================] - 72s 23ms/step
              precision    recall  f1-score   support

           0       0.98      0.38      0.54     71451
           1       0.39      0.98      0.55     28455

    accuracy        